# Notebook 1 — Setup & Ingest Pipeline

*Hands-on time: ~40 minutes*

In this notebook you will:
1. Verify the environment (ES, Kibana, BIDS datasets)
2. Explore multiple BIDS datasets with `pybids`
3. Design an ElasticSearch index mapping
4. Generate text embeddings from metadata
5. Bulk-index everything into ES

**Prerequisites:** Complete the setup in `docs/00-overview.md` and read chapters 1–3.

---
## 1. Verify the Environment

In [1]:
import os
from elasticsearch import Elasticsearch

ES_HOST = os.environ.get("ES_HOST", "http://localhost:9200")
client = Elasticsearch(ES_HOST, request_timeout=120)

# Check connection
info = client.info()
print(f"Cluster name: {info['cluster_name']}")
print(f"ES version:   {info['version']['number']}")

health = client.cluster.health()
print(f"Status:        {health['status']}")
assert health['status'] in ('green', 'yellow'), "ES cluster is not healthy!"

Cluster name: docker-cluster
ES version:   9.3.0
Status:        green


In [2]:
from pathlib import Path

DATA_DIR = Path("./data")
assert DATA_DIR.exists(
), f"Data directory not found at {DATA_DIR.resolve()}. Run scripts/download_dataset.sh first."

# List available BIDS datasets
datasets = sorted([
    d for d in DATA_DIR.iterdir()
    if d.is_dir() and (d / "dataset_description.json").exists()
])

print(f"Found {len(datasets)} BIDS dataset(s):\n")
for ds in datasets:
    nifti_count = len(list(ds.rglob("*.nii.gz")))
    print(f"  {ds.name:25s} ({nifti_count} NIfTI files)")

assert len(
    datasets) > 0, "No BIDS datasets found! Run scripts/download_dataset.sh first."

Found 82 BIDS dataset(s):

  2d_mb_pcasl               (4 NIfTI files)
  7t_trt                    (439 NIfTI files)
  asl001                    (2 NIfTI files)
  asl002                    (3 NIfTI files)
  asl003                    (3 NIfTI files)
  asl004                    (4 NIfTI files)
  asl005                    (3 NIfTI files)
  atlas-4S                  (26 NIfTI files)
  atlas-AAL                 (2 NIfTI files)
  atlas-Destrieux           (2 NIfTI files)
  atlas-DiFuMo              (6 NIfTI files)
  atlas-HOSPA               (15 NIfTI files)
  atlas-HarvardOxford       (3 NIfTI files)
  atlas-Juelich             (3 NIfTI files)
  atlas-Schaefer            (4 NIfTI files)
  atlas-Talairach           (6 NIfTI files)
  atlas-suit                (7 NIfTI files)
  ds000001-fmriprep         (128 NIfTI files)
  ds000117                  (395 NIfTI files)
  ds000246                  (1 NIfTI files)
  ds000247                  (5 NIfTI files)
  ds000248                  (6 NIfTI file

---
## 2. Explore the BIDS Datasets with pybids

`BIDSLayout` indexes a dataset and provides a query API that handles the
BIDS inheritance principle automatically. We'll load all the datasets and
then inspect a file from the metadata-richest one.

In [3]:
from bids import BIDSLayout

# Load layouts for all datasets
layouts = {}
for ds_path in datasets:
    layout = BIDSLayout(str(ds_path), validate=False)
    layouts[ds_path.name] = layout
    subjects = layout.get_subjects()
    print(f"\n{ds_path.name}:")
    print(f"  Subjects:    {subjects[:5]}{'...' if len(subjects) > 5 else ''}")
    print(f"  Data types:  {layout.get_datatypes()}")
    print(f"  Suffixes:    {layout.get_suffixes()}")
    print(f"  Tasks:       {layout.get_tasks()}")
    print(f"  NIfTI files: {len(layout.get(extension='.nii.gz'))}")

total_files = sum(len(l.get(extension=".nii.gz")) for l in layouts.values())
print(f"\nTotal across all datasets: {total_files} NIfTI files")


2d_mb_pcasl:
  Subjects:    ['1']
  Data types:  ['anat', 'fmap', 'perf']
  Suffixes:    ['asl', 'aslcontext', 'description', 'epi', 'T1w']
  Tasks:       []
  NIfTI files: 4

7t_trt:
  Subjects:    ['01', '02', '03', '04', '05']...
  Data types:  ['anat', 'fmap', 'func']
  Suffixes:    ['bold', 'description', 'magnitude1', 'magnitude2', 'participants', 'phasediff', 'physio', 'scans', 'sessions', 'T1map', 'T1w']
  Tasks:       ['rest']
  NIfTI files: 439

asl001:
  Subjects:    ['Sub103']
  Data types:  ['anat', 'perf']
  Suffixes:    ['asl', 'aslcontext', 'asllabeling', 'description', 'T1w']
  Tasks:       []
  NIfTI files: 2

asl002:
  Subjects:    ['Sub103']
  Data types:  ['anat', 'perf']
  Suffixes:    ['asl', 'aslcontext', 'asllabeling', 'description', 'm0scan', 'T1w']
  Tasks:       []
  NIfTI files: 3

asl003:
  Subjects:    ['Sub1']
  Data types:  ['anat', 'perf']
  Suffixes:    ['asl', 'aslcontext', 'asllabeling', 'description', 'm0scan', 'T1w']
  Tasks:       []
  NIfTI fil

In [4]:
# Pick a file from the metadata-richest dataset for exploration
# ds000117 has the richest JSON sidecars (50+ fields)
for ds_name in ["ds000117", "eeg_rest_fmri", "ds001"]:
    if ds_name in layouts:
        bold_files = layouts[ds_name].get(suffix="bold", extension=".nii.gz")
        if bold_files:
            example_file = bold_files[0]
            example_layout = layouts[ds_name]
            example_dataset = ds_name
            break

print(f"Example from dataset '{example_dataset}':")
print(f"File: {example_file.filename}\n")

print("Entities extracted from filename:")
for key, val in example_file.entities.items():
    print(f"  {key}: {val}")

print(f"\nResolved metadata from JSON sidecar(s):")
meta = example_layout.get_metadata(example_file.path)
for key, val in sorted(meta.items()):
    print(f"  {key}: {val}")

Example from dataset 'ds000117':
File: sub-01_ses-mri_task-facerecognition_run-01_bold.nii.gz

Entities extracted from filename:
  AcquisitionMatrixPE: 64
  AcquisitionNumber: 1
  BandwidthPerPixelPhaseEncode: 30.637
  BaseResolution: 64
  ConversionSoftware: dcm2niix
  ConversionSoftwareVersion: v1.0.20171215 GCC4.4.7
  DerivedVendorReportedEchoSpacing: 0.000510004
  DeviceSerialNumber: 35119
  DwellTime: 3.5e-06
  EchoTime: 0.03
  EffectiveEchoSpacing: 0.000510004
  FlipAngle: 78
  ImageComments: V
  ImageOrientationPatientDICOM: [0.999864, 0.005398, 0.015583, 0, 0.94491, -0.32733]
  ImageType: ['ORIGINAL', 'PRIMARY', 'M', 'ND', 'MOSAIC']
  InPlanePhaseEncodingDirectionDICOM: COL
  InstitutionAddress: 15 Chaucer Road, Cambridge CB2 7EF, UK
  InstitutionName: MRC Cognition and Brain Sciences Unit
  MRAcquisitionType: 2D
  MagneticFieldStrength: 3
  Manufacturer: Siemens
  ManufacturersModelName: TrioTim
  Modality: MR
  NumberOfVolumesDiscardedByScanner: 3
  NumberOfVolumesDiscardedBy

In [9]:
import pandas as pd

# Load participant demographics from all datasets
all_participants = {}
for ds_path in datasets:
    ds_name = ds_path.name
    pfile = ds_path / "participants.tsv"
    if pfile.exists():
        df = pd.read_csv(pfile, sep="\t")
        all_participants[ds_name] = df
        print(f"{ds_name:25s} {len(df):3d} participants — columns: {list(df.columns)}")
    else:
        print(f"{ds_name:25s}  no participants.tsv")

# Show one example
if all_participants:
    first_key = list(all_participants.keys())[0]
    print(f"\n{first_key} participants.tsv:")
    display(all_participants[first_key].head(5))

2d_mb_pcasl                no participants.tsv
7t_trt                     22 participants — columns: ['participant_id', 'sex', 'age_at_first_scan_years', 'number_of_scans_before', 'handedness']
asl001                     no participants.tsv
asl002                     no participants.tsv
asl003                     no participants.tsv
asl004                     no participants.tsv
asl005                     no participants.tsv
atlas-4S                   no participants.tsv
atlas-AAL                  no participants.tsv
atlas-Destrieux            no participants.tsv
atlas-DiFuMo               no participants.tsv
atlas-HOSPA                no participants.tsv
atlas-HarvardOxford        no participants.tsv
atlas-Juelich              no participants.tsv
atlas-Schaefer             no participants.tsv
atlas-Talairach            no participants.tsv
atlas-suit                 no participants.tsv
ds000001-fmriprep          no participants.tsv
ds000117                   17 participants — columns: 

,participant_id,sex,age_at_first_scan_years,number_of_scans_before,handedness
0,sub-01,F,29,17,100
1,sub-02,F,23,6,100
2,sub-03,M,25,18,86
3,sub-04,M,26,8,100
4,sub-05,M,27,28,-84


---
## 3. Design the ElasticSearch Index Mapping

We define the schema explicitly so ES knows how to index each field optimally.
The mapping reflects the metadata fields we explored above.

In [ ]:
INDEX_NAME = "neuroimaging"
EMBEDDING_DIMS = 768  # all-mpnet-base-v2 output dimensionality

INDEX_SETTINGS = {
    "number_of_replicas": 0   # single-node cluster — no replicas needed
}

INDEX_MAPPINGS = {
    "properties": {
        # === Dataset identifier ===
        "dataset":   {"type": "keyword"},

        # === Filename-derived entities ===
        "subject":   {"type": "keyword"},
        "session":   {"type": "keyword"},
        "task":      {"type": "keyword"},
        "run":       {"type": "keyword"},
        "suffix":    {"type": "keyword"},
        "datatype":  {"type": "keyword"},

        # === Participant demographics ===
        "age":       {"type": "float"},
        "sex":       {"type": "keyword"},

        # === JSON sidecar metadata (numeric) ===
        "RepetitionTime":        {"type": "float"},
        "EchoTime":              {"type": "float"},
        "FlipAngle":             {"type": "float"},
        "MagneticFieldStrength": {"type": "float"},
        "SliceThickness":        {"type": "float"},
        "InversionTime":         {"type": "float"},

        # === JSON sidecar metadata (categorical) ===
        "Manufacturer":           {"type": "keyword"},
        "ManufacturersModelName": {"type": "keyword"},
        "InstitutionName":        {"type": "keyword"},
        "PhaseEncodingDirection": {"type": "keyword"},
        "PulseSequenceType":      {"type": "keyword"},
        "MRAcquisitionType":      {"type": "keyword"},
        "ScanningSequence":       {"type": "keyword"},
        "BodyPart":               {"type": "keyword"},
        "ReceiveCoilName":        {"type": "keyword"},

        # === JSON sidecar metadata (text — full-text searchable) ===
        "TaskName":          {"type": "text"},
        "TaskDescription":   {"type": "text"},
        "SeriesDescription": {"type": "text"},
        "ProtocolName":      {"type": "text"},

        # === Computed fields ===
        # concatenated summary for BM25
        "description_text":   {"type": "text"},
        "metadata_embedding": {                       # dense vector for kNN
            "type": "dense_vector",
            "dims": EMBEDDING_DIMS,  # 768 (all-mpnet-base-v2)
            "similarity": "cosine",
            "index_options": {"type": "int8_hnsw"}
        },

        # === File reference ===
        "bids_path": {"type": "keyword"},

        # === Computed enrichment fields ===
        # from dataset_description.json
        "study_description": {"type": "text"},
        # functional/structural/diffusion/…
        "modality_group":    {"type": "keyword"}
    }
}

print("Mapping defined with fields:")
for field, config in INDEX_MAPPINGS["properties"].items():
    ftype = config.get("type", config.get("type", "?"))
    print(f"  {field:30s} → {ftype}")

Mapping defined with fields:
  dataset                        → keyword
  subject                        → keyword
  session                        → keyword
  task                           → keyword
  run                            → keyword
  suffix                         → keyword
  datatype                       → keyword
  age                            → float
  sex                            → keyword
  RepetitionTime                 → float
  EchoTime                       → float
  FlipAngle                      → float
  MagneticFieldStrength          → float
  SliceThickness                 → float
  InversionTime                  → float
  Manufacturer                   → keyword
  ManufacturersModelName         → keyword
  InstitutionName                → keyword
  PhaseEncodingDirection         → keyword
  PulseSequenceType              → keyword
  MRAcquisitionType              → keyword
  ScanningSequence               → keyword
  BodyPart                       → keyw

In [6]:
# Delete existing index if present (for re-runs), then create
if client.indices.exists(index=INDEX_NAME):
    client.indices.delete(index=INDEX_NAME)
    print(f"Deleted existing index '{INDEX_NAME}'")

client.indices.create(
    index=INDEX_NAME, settings=INDEX_SETTINGS, mappings=INDEX_MAPPINGS)
print(f"Created index '{INDEX_NAME}' with explicit mapping")

Deleted existing index 'neuroimaging'
Created index 'neuroimaging' with explicit mapping


---
## 4. Generate Text Embeddings

We encode a concatenated metadata string for each scan. The embedding model
captures semantic relationships (e.g., \"T1w\" and \"structural MRI\" will be
close in vector space).

**First run:** The model (~80 MB) will be downloaded automatically.

In [7]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-mpnet-base-v2", device="cpu")
print(
    f"Model: all-mpnet-base-v2 | Output dims: {model.get_sentence_embedding_dimension()} (expect 768)")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model: all-mpnet-base-v2 | Output dims: 768 (expect 768)


In [ ]:
import json as _json

SUFFIX_DESCRIPTIONS = {
    # --- Structural anatomical MRI ---
    "T1w":        "T1-weighted anatomical structural MRI",
    "T2w":        "T2-weighted anatomical structural MRI",
    "FLAIR":      "fluid-attenuated inversion recovery MRI",
    "FLASH":      "fast low-angle shot gradient echo MRI",
    "inplaneT2":  "in-plane T2-weighted anatomical MRI",
    "UNIT1":      "uniform T1 image from MP2RAGE",
    "defacemask": "binary de-face mask for anatomical anonymization",
    # --- Functional MRI ---
    "bold":       "BOLD functional MRI Blood Oxygen Level Dependent",
    "boldref":    "BOLD reference volume for functional alignment",
    "sbref":      "single-band reference image for multiband fMRI",
    # --- Diffusion ---
    "dwi":        "diffusion-weighted imaging DWI DTI tractography white matter",
    # --- Fieldmaps ---
    "phasediff":  "phase-difference fieldmap distortion correction B0",
    "magnitude1": "magnitude image 1 for fieldmap estimation",
    "magnitude2": "magnitude image 2 for fieldmap estimation",
    "magnitude":  "magnitude image for fieldmap estimation",
    "fieldmap":   "directly measured B0 fieldmap",
    "epi":        "EPI fieldmap blip-up blip-down distortion correction",
    # --- Quantitative MRI / relaxometry ---
    "T1map":      "quantitative T1 relaxation time map qMRI",
    "T2map":      "quantitative T2 relaxation time map qMRI",
    "T2starmap":  "quantitative T2* relaxation time map qMRI",
    "R1map":      "quantitative R1 relaxation rate map",
    "R2starmap":  "quantitative R2* relaxation rate map",
    "PDmap":      "proton density map",
    "M0map":      "equilibrium magnetization M0 map",
    "MTsat":      "magnetization transfer saturation map",
    "MTRmap":     "magnetization transfer ratio map",
    "MWFmap":     "myelin water fraction map",
    "Chimap":     "quantitative susceptibility chi map QSM",
    "TB1map":     "transmit B1 field map",
    "VFA":        "variable flip-angle images T1 mapping",
    "IRT1":       "inversion-recovery T1 mapping",
    "MTS":        "magnetization transfer saturation weighted image",
    "TB1DAM":     "double-angle method B1 transmit field mapping",
    "TB1TFL":     "turbo-flash B1 transmit field mapping",
    "TB1AFI":     "actual flip-angle imaging B1 transmit field mapping",
    "TB1SRGE":    "saturation-recovery gradient echo B1 mapping",
    "MESE":       "multi-echo spin-echo T2 mapping",
    "MEGRE":      "multi-echo gradient-echo T2* mapping",
    # --- Segmentation / derived ---
    "dseg":       "discrete segmentation label map atlas parcellation",
    "probseg":    "probabilistic segmentation map",
    "mask":       "binary brain mask ROI",
    "register":   "registration transform or registered image",
    # --- Spectroscopy ---
    "svs":        "single-voxel MR spectroscopy MRS",
    "mrsref":     "MR spectroscopy reference scan",
    "mrsi":       "MR spectroscopic imaging chemical shift imaging",
    # --- PET ---
    "pet":        "positron emission tomography PET nuclear medicine tracer",
    # --- ASL / perfusion ---
    "asl":        "arterial spin labeling perfusion imaging cerebral blood flow",
    "m0scan":     "M0 calibration scan for arterial spin labeling",
}

# Broad modality groups for faceting and enriched description
MODALITY_GROUPS = {
    "bold": "functional", "sbref": "functional", "boldref": "functional",
    "T1w": "structural", "T2w": "structural", "FLAIR": "structural",
    "FLASH": "structural", "inplaneT2": "structural", "UNIT1": "structural",
    "dwi": "diffusion",
    "phasediff": "fieldmap", "magnitude1": "fieldmap", "magnitude2": "fieldmap",
    "magnitude": "fieldmap", "fieldmap": "fieldmap", "epi": "fieldmap",
    "T1map": "quantitative", "T2map": "quantitative", "T2starmap": "quantitative",
    "MTsat": "quantitative", "MTRmap": "quantitative", "Chimap": "quantitative",
    "TB1map": "quantitative", "VFA": "quantitative", "IRT1": "quantitative",
    "MESE": "quantitative", "MEGRE": "quantitative",
    "asl": "perfusion", "m0scan": "perfusion",
    "pet": "pet", "svs": "spectroscopy", "mrsi": "spectroscopy",
}


def _is_valid(value) -> bool:
    """Return True if value is non-None, non-NaN, and non-empty."""
    if value is None:
        return False
    try:
        if isinstance(value, float) and value != value:  # NaN
            return False
    except TypeError:
        pass
    if isinstance(value, str) and value.strip().lower() in ("", "nan", "n/a"):
        return False
    return True


def build_description_text(entities: dict, metadata: dict, participant_info: dict,
                           study_description: str = "") -> str:
    """
    Build a rich, prose-style description for BM25 and embedding.

    Compared to the original version this function:
    - Expands abbreviations inline ('3 Tesla (3T)' instead of just '3T')
    - Includes the full TaskDescription (up to 300 chars)
    - Adds the study-level name/description from dataset_description.json
    - Uses 'acquired at …' and 'receive coil: …' for more natural language
    """
    parts = []
    suffix = entities.get("suffix", "")
    modality = MODALITY_GROUPS.get(suffix, "")

    # Modality + suffix
    suffix_desc = SUFFIX_DESCRIPTIONS.get(suffix, suffix)
    parts.append(
        f"{suffix_desc} ({modality} MRI)" if modality else suffix_desc)

    # Task / paradigm
    task_name = metadata.get("TaskName", "") or entities.get("task", "")
    if task_name:
        parts.append(f"task: {task_name}")
    task_desc = metadata.get("TaskDescription", "")
    if task_desc:
        parts.append(task_desc[:300])

    # Scanner
    field_strength = metadata.get("MagneticFieldStrength")
    if field_strength:
        tesla_map = {1.5: "1.5 Tesla", 3.0: "3 Tesla", 3: "3 Tesla",
                     7.0: "7 Tesla", 7: "7 Tesla"}
        tesla_str = tesla_map.get(field_strength, f"{field_strength} Tesla")
        parts.append(f"{tesla_str} ({field_strength}T) MRI scanner")

    manufacturer = metadata.get("Manufacturer", "")
    model_name = metadata.get("ManufacturersModelName", "")
    if manufacturer:
        parts.append(
            f"{manufacturer} scanner{' model ' + model_name if model_name else ''}")
    inst = metadata.get("InstitutionName", "")
    if inst:
        parts.append(f"acquired at {inst}")

    # Acquisition parameters
    acq_type = metadata.get("MRAcquisitionType", "")
    if acq_type:
        parts.append(f"{acq_type} acquisition")

    pulse_seq = metadata.get("PulseSequenceType", "")
    scanning_seq = metadata.get("ScanningSequence", "")
    if pulse_seq:
        parts.append(f"pulse sequence: {pulse_seq}")
    elif scanning_seq:
        parts.append(f"scanning sequence: {scanning_seq}")

    tr = metadata.get("RepetitionTime")
    if tr is not None:
        parts.append(f"RepetitionTime {tr}s")
    te = metadata.get("EchoTime")
    if te is not None:
        parts.append(f"EchoTime {te}s")
    ti = metadata.get("InversionTime")
    if ti is not None:
        parts.append(f"InversionTime {ti}s")
    fa = metadata.get("FlipAngle")
    if fa is not None:
        parts.append(f"FlipAngle {fa} degrees")

    body_part = metadata.get("BodyPart", metadata.get("BodyPartExamined", ""))
    if body_part:
        parts.append(f"body part: {body_part}")
    coil = metadata.get("ReceiveCoilName", "")
    if coil:
        parts.append(f"receive coil: {coil}")

    series_desc = metadata.get("SeriesDescription", "")
    if series_desc:
        parts.append(f"series: {series_desc}")
    protocol = metadata.get("ProtocolName", "")
    if protocol and protocol != series_desc:
        parts.append(f"protocol: {protocol}")

    # Study-level context
    if study_description:
        parts.append(study_description[:200])

    # Demographics
    age = participant_info.get("age")
    sex = participant_info.get("sex")
    if _is_valid(age):
        parts.append(f"participant age {age}")
    if _is_valid(sex):
        parts.append(f"participant sex {sex}")

    return " | ".join(parts)


# ── Demo: show what a description text looks like after the upgrade ──────────
demo_entities = example_file.entities
demo_meta = example_layout.get_metadata(example_file.path)
demo_participant = {}
if example_dataset in all_participants:
    subj_id = f"sub-{demo_entities.get('subject', '')}"
    row_match = all_participants[example_dataset]
    row_match = row_match[row_match["participant_id"] == subj_id]
    if not row_match.empty:
        demo_participant = row_match.iloc[0].to_dict()

# Also grab study description from dataset_description.json
study_desc_path = Path(f"./data/{example_dataset}/dataset_description.json")
study_desc = ""
if study_desc_path.exists():
    try:
        dd = _json.loads(study_desc_path.read_text())
        study_desc = dd.get("Name", "") + " " + dd.get("HowToAcknowledge", "")
        study_desc = study_desc.strip()
    except Exception:
        pass

demo_text = build_description_text(
    demo_entities, demo_meta, demo_participant, study_desc)
print(f"Enriched description text:\n  {demo_text}\n")

demo_embedding = model.encode(demo_text)
print(
    f"Embedding shape: {demo_embedding.shape}  (expect (768,) for all-mpnet-base-v2)")

Enriched description text:
  BOLD functional MRI Blood Oxygen Level Dependent (functional MRI) | task: facerecognition | 3 Tesla (3T) MRI scanner | Siemens scanner model TrioTim | acquired at MRC Cognition and Brain Sciences Unit | 2D acquisition | pulse sequence: EPI | RepetitionTime 2s | EchoTime 0.03s | FlipAngle 78 degrees | receive coil: HeadMatrix | series: CBU_DWEPI_BOLD210 | Multisubject, multimodal face processing Cite this paper: https://www.ncbi.nlm.nih.gov/pubmed/25977808 and consider including the following message: 'This data was obtained from the OpenNeuro database | participant age 31.0 | participant sex M

Embedding shape: (768,)  (expect (768,) for all-mpnet-base-v2)


---
## 5. Build & Run the Ingestion Pipeline

We iterate over every NIfTI file across **all datasets**, extract metadata,
generate embeddings, and bulk-index into ElasticSearch. Each document carries a
`dataset` field so we can filter and aggregate by dataset later.

In [ ]:
from elasticsearch import helpers
from tqdm.auto import tqdm
import numpy as np


def safe_float(value):
    """Convert a value to float, returning None for non-numeric values (including NaN).
    Handles pybids PaddedInt, string ages like '89+', pandas NaN, etc."""
    if value is None:
        return None
    try:
        result = float(value)
        if result != result:  # NaN check
            return None
        return result
    except (ValueError, TypeError):
        return None


def safe_str(value, default=""):
    """Convert a value to str, returning default for None/NaN."""
    if value is None:
        return default
    try:
        if isinstance(value, float) and value != value:  # NaN
            return default
    except TypeError:
        pass
    s = str(value)
    if s.lower() == "nan":
        return default
    return s


def generate_all_documents(layouts, all_participants, model):
    """Yield ES documents for every NIfTI file across all BIDS datasets."""
    for ds_name, layout in layouts.items():
        nifti_files = layout.get(extension=".nii.gz")
        dataset_root = Path(layout.root).resolve()

        # Build participant lookup for this dataset
        participant_lookup = {}
        if ds_name in all_participants:
            for _, row in all_participants[ds_name].iterrows():
                participant_lookup[row["participant_id"]] = row.to_dict()

        # Collect file metadata
        file_data = []
        for bf in nifti_files:
            entities = bf.entities
            metadata = layout.get_metadata(bf.path)
            subj_id = f"sub-{entities.get('subject', '')}"
            participant_info = participant_lookup.get(subj_id, {})
            # Read study-level description once per dataset
            study_desc_path_local = Path(
                layout.root) / "dataset_description.json"
            study_description_local = ""
            if study_desc_path_local.exists():
                try:
                    _dd = _json.loads(study_desc_path_local.read_text())
                    study_description_local = (
                        _dd.get("Name", "") + " " + _dd.get("HowToAcknowledge", "")).strip()
                except Exception:
                    pass

            desc_text = build_description_text(
                entities, metadata, participant_info,
                study_description=study_description_local)

            file_data.append({
                "entities": entities,
                "metadata": metadata,
                "participant_info": participant_info,
                "desc_text": desc_text,
                "study_description": study_description_local,
                "bids_path": str(Path(bf.path).resolve().relative_to(dataset_root)),
            })

        if not file_data:
            print(f"  {ds_name}: no NIfTI files found — skipping")
            continue

        # Batch-encode all description texts for this dataset
        desc_texts = [fd["desc_text"] for fd in file_data]
        print(f"  {ds_name}: encoding {len(desc_texts)} descriptions...")
        embeddings = model.encode(
            desc_texts, show_progress_bar=True, batch_size=32)

        # Yield ES documents
        for fd, embedding in zip(file_data, embeddings):
            entities = fd["entities"]
            metadata = fd["metadata"]
            participant_info = fd["participant_info"]

            doc = {
                # Dataset
                "dataset":   ds_name,

                # Filename entities
                "subject":   str(entities.get("subject", "")),
                "session":   str(entities.get("session", "")),
                "task":      str(entities.get("task", "")),
                "run":       str(entities.get("run", "")),
                "suffix":    str(entities.get("suffix", "")),
                "datatype":  str(entities.get("datatype", "")),

                # Demographics
                "age":       safe_float(participant_info.get("age")),
                "sex":       safe_str(participant_info.get("sex")),

                # Sidecar metadata (numeric)
                "RepetitionTime":        safe_float(metadata.get("RepetitionTime")),
                "EchoTime":              safe_float(metadata.get("EchoTime")),
                "FlipAngle":             safe_float(metadata.get("FlipAngle")),
                "MagneticFieldStrength": safe_float(metadata.get("MagneticFieldStrength")),
                "SliceThickness":        safe_float(metadata.get("SliceThickness")),
                "InversionTime":         safe_float(metadata.get("InversionTime")),

                # Sidecar metadata (categorical)
                "Manufacturer":           metadata.get("Manufacturer"),
                "ManufacturersModelName": metadata.get("ManufacturersModelName"),
                "InstitutionName":        metadata.get("InstitutionName"),
                "PhaseEncodingDirection": metadata.get("PhaseEncodingDirection"),
                "PulseSequenceType":      metadata.get("PulseSequenceType"),
                "MRAcquisitionType":      metadata.get("MRAcquisitionType"),
                "ScanningSequence":       metadata.get("ScanningSequence"),
                "BodyPart":               metadata.get("BodyPart", metadata.get("BodyPartExamined")),
                "ReceiveCoilName":        metadata.get("ReceiveCoilName"),

                # Sidecar metadata (text)
                "TaskName":          metadata.get("TaskName"),
                "TaskDescription":   metadata.get("TaskDescription"),
                "SeriesDescription": metadata.get("SeriesDescription"),
                "ProtocolName":      metadata.get("ProtocolName"),

                # Computed
                "description_text":   fd["desc_text"],
                "metadata_embedding": embedding.tolist(),
                "study_description":  fd.get("study_description", ""),
                "modality_group":     MODALITY_GROUPS.get(str(fd["entities"].get("suffix", "")), "other"),

                # File path
                "bids_path": fd["bids_path"],
            }

            yield {"_index": INDEX_NAME, "_source": doc}


print(
    f"Starting ingestion of {len(layouts)} dataset(s) into index '{INDEX_NAME}'...")

Starting ingestion of 82 dataset(s) into index 'neuroimaging'...


In [12]:
# Run the bulk ingestion across all datasets
success_count, errors = helpers.bulk(
    client,
    generate_all_documents(layouts, all_participants, model),
    raise_on_error=False,   # log failures instead of aborting entire run
    refresh="wait_for"      # ensure docs are searchable immediately after
)

print(f"\nIngestion complete!")
print(f"  Documents indexed: {success_count}")
if errors:
    print(f"  Errors:            {len(errors)}")
    for err in errors[:5]:
        print(f"    ⚠ {err}")
else:
    print(f"  Errors:            0")

  2d_mb_pcasl: encoding 4 descriptions...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

  7t_trt: encoding 439 descriptions...


Batches:   0%|          | 0/14 [00:00<?, ?it/s]

  asl001: encoding 2 descriptions...


...

  synthetic: no NIfTI files found — skipping
  volume_timing: encoding 6 descriptions...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

  xeeg_hed_score: encoding 1 descriptions...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]


Ingestion complete!
  Documents indexed: 4423
  Errors:            0


---
## 6. Verify the Index

In [13]:
# Count documents
count = client.count(index=INDEX_NAME)
print(f"Total documents in '{INDEX_NAME}': {count['count']}")
assert count['count'] > 0, "No documents were indexed!"

# Retrieve one document to inspect
sample = client.search(
    index=INDEX_NAME,
    query={"match_all": {}},
    size=1,
    fields=["dataset", "subject", "suffix", "task", "RepetitionTime", "Manufacturer",
            "MagneticFieldStrength", "InstitutionName", "description_text", "bids_path"]
)

hit = sample["hits"]["hits"][0]
print(f"\nSample document (fields):")
for field, values in hit.get("fields", {}).items():
    print(f"  {field}: {values}")

Total documents in 'neuroimaging': 4423

Sample document (fields):
  RepetitionTime: [2.5]
  task: ['']
  MagneticFieldStrength: [3.0]
  subject: ['1']
  bids_path: ['sub-1/anat/sub-1_T1w.nii.gz']
  Manufacturer: ['Siemens']
  description_text: ['T1-weighted anatomical structural MRI (structural MRI) | 3 Tesla (3T) MRI scanner | Siemens scanner model Prisma_fit | 3D acquisition | scanning sequence: GR\\IR | RepetitionTime 2.5s | EchoTime 0.00222s | InversionTime 1s | FlipAngle 8 degrees | body part: BRAIN | receive coil: Head_32 | series: T1w_MPR | ASL_Siemens_MultiPLD_2DMB_PCASL If you reference this dataset in your publications, please acknowledge its authors.']
  suffix: ['T1w']
  dataset: ['2d_mb_pcasl']


In [14]:
# Aggregation: documents per dataset, then per suffix within each dataset
agg_result = client.search(
    index=INDEX_NAME,
    size=0,
    aggs={
        "by_dataset": {
            "terms": {"field": "dataset"},
            "aggs": {
                "by_suffix": {
                    "terms": {"field": "suffix"}
                }
            }
        }
    }
)

print("Documents by dataset and suffix:\n")
for ds_bucket in agg_result["aggregations"]["by_dataset"]["buckets"]:
    print(f"  {ds_bucket['key']} ({ds_bucket['doc_count']} docs):")
    for suffix_bucket in ds_bucket["by_suffix"]["buckets"]:
        print(
            f"    {suffix_bucket['key']:15s} {suffix_bucket['doc_count']:4d}")
    print()

Documents by dataset and suffix:

  7t_trt (439 docs):
    bold             132
    magnitude1        88
    phasediff         88
    magnitude2        87
    T1map             22
    T1w               22

  ds000117 (395 docs):
    FLASH            224
    bold             144
    T1w               16
    dwi               11

  ds108 (238 docs):
    bold             204
    T1w               34

  ds210 (225 docs):
    bold             225

  ds006 (220 docs):
    bold             164
    T1w               28
    inplaneT2         28

  ds110 (216 docs):
    bold             180
    T1w               18
    inplaneT2         18

  ds009 (192 docs):
    bold             144
    T1w               24
    inplaneT2         24

  ds007 (158 docs):
    bold             118
    T1w               20
    inplaneT2         20

  ds113b (156 docs):
    bold             156

  ds107 (147 docs):
    bold              98
    T1w               49



---
## Summary

You've now:
- ✅ Connected to ElasticSearch and verified the cluster
- ✅ Explored **all BIDS-examples datasets** programmatically with `pybids`
- ✅ Created an ES index with explicit mappings for structured metadata + dense vectors
- ✅ Generated embeddings from concatenated metadata using `sentence-transformers`
- ✅ Bulk-indexed documents from all datasets, each tagged with a `dataset` field
- ✅ Verified the index with counts and per-dataset aggregations

**Next:** Open **Notebook 2** to start querying this data with keyword and filtered search.